In [44]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
GK_DATA = Path("GK_Data")
def load_keeper_match_kpis(player_id, match_dirs_str):
    """Load player_kpis for a keeper from all his matches."""
    all_kpis = []
    if not isinstance(match_dirs_str, str) or not match_dirs_str:
        return all_kpis

    for match_dir in match_dirs_str.split("|"):
        match_dir = match_dir.strip()
        if not match_dir:
            continue
        for cs_dir in (GK_DATA / "competitions").iterdir():
            if not cs_dir.is_dir():
                continue
            pkpi_path = cs_dir / match_dir / "player_kpis.json"
            if pkpi_path.exists():
                with open(pkpi_path) as f:
                    data = json.load(f).get("data", {})
                for side in ["squadHome", "squadAway"]:
                    for player in data.get(side, {}).get("players", []):
                        if (
                            player["id"] == player_id
                            and player.get("position") == "GOALKEEPER"
                            and player.get("matchShare", 0) >= 0.5
                        ):
                            kpis = {k["kpiId"]: k["value"] for k in player.get("kpis", [])}
                            kpis["matchShare"] = player["matchShare"]
                            kpis["playDuration"] = player.get("playDuration", 0)
                            all_kpis.append(kpis)
                break  
    return all_kpis

dataset = pd.read_csv(GK_DATA / "gk_dataset_final.csv")

with open(GK_DATA / "player_kpi_definitions.json") as f:
    kpi_defs = {d["id"]: d["name"] for d in json.load(f).get("data", [])}

all_match_rows = []

for _, row in dataset.iterrows():
    match_kpis = load_keeper_match_kpis(row["playerId"], row["origin_match_dirs"])
    
    if match_kpis:
        for match_data in match_kpis:
            entry = row.drop(['origin_match_dirs', 'current_match_dirs']).to_dict()
            
            for kpi_id, value in match_data.items():
                kpi_name = kpi_defs.get(kpi_id, f"KPI_{kpi_id}")
                entry[kpi_name] = value
            
            all_match_rows.append(entry)

df_long = pd.DataFrame(all_match_rows)
print(f"Done! Created a dataset with {len(df_long)} total match-performance rows.")

Building long-form dataset...
Done! Created a dataset with 26127 total match-performance rows.


In [8]:
df_long.sample(5)

,playerId,name,age,birthdate,status,origin_team,origin_comp,origin_season,origin_median,origin_matches,...,LOST_GROUND_DUELS_IN_PITCH_POSITION_OPPONENT_BOX,ASSISTS_BY_ACTION_SAVE,ASSISTS_AT_PHASE_SECOND_BALL,BYPASSED_OPPONENTS_FROM_PACKING_ZONE_WR,BYPASSED_OPPONENTS_NUMBER_FROM_PACKING_ZONE_WR,BALL_LOSS_REMOVED_TEAMMATES_FROM_PACKING_ZONE_WL,BYPASSED_DEFENDERS_RECEIVING_TO_PITCH_POSITION_FIRST_THIRD,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_LANE_RIGHT_WING,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_OPPONENT_BOX,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_PITCH_POSITION_OPPONENT_BOX
1061,29405,Lazar Carević,26.0,1999-03-16,PLAYS,FK Vojvodina Novi Sad,super_liga_srbije,2023-2024,0.349178,33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3615,9586,Manuel Riemann,37.0,1988-09-09,DROPPED,VfL Bochum,bundesliga_1,2023-2024,0.826471,33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24186,19646,Miguel Silva,30.0,1995-04-07,STAYED,Beitar Jerusalem,ligat_haal,2024-2025,0.500000,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4066,9149,Nick Olij,30.0,1995-08-01,DROPPED,Sparta Rotterdam,eredivisie,2023-2024,0.691707,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13369,16517,Shusaku Nishikawa,39.0,1986-06-18,STAYED,Urawa Red Diamonds,j1_league,2024,0.614976,208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# 1. Define the KPIs we want to analyze (the "Real" ones we found earlier)
kpi_cols = [c for c in df_long.columns if c.startswith('KPI_') or c.isupper()]

# 2. Aggregate by PlayerID
# We take the MEAN of performance, but the FIRST of metadata (since age/status don't change per match)
df_players = df_long.groupby('playerId').agg({
    'status': 'first',
    'age': 'first',
    'origin_comp': 'first',
    **{kpi: 'mean' for kpi in kpi_cols} # Average performance across all origin matches
}).reset_index()

# 3. Create our "Riser" target
# We want to distinguish 'PLAYS' (The Risers) from everyone else
df_players['is_riser'] = (df_players['status'] == 'PLAYS').astype(int)

print(f"Aggregated {len(df_long)} matches into {len(df_players)} unique player profiles.")

Aggregated 26127 matches into 627 unique player profiles.


/tmp/ipykernel_5039/3332150904.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()
/tmp/ipykernel_5039/3332150904.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_players['is_riser'] = (df_players['status'] == 'PLAYS').astype(int)


In [23]:
def calculate_slope(series):
    if len(series) < 3: return 0 # Need at least 3 matches to see a trend
    return np.polyfit(range(len(series)), series, 1)[0]

# Add 'Trend' features for your top KPIs
df_trends = df_long.groupby('playerId')['BYPASSED_OPPONENTS'].apply(calculate_slope).reset_index()
df_trends.columns = ['playerId', 'BYPASSED_OPPONENTS_TREND']

# Merge this back to your player profile
df_players = df_players.merge(df_trends, on='playerId')

In [24]:
# Drop non-numeric and leakage
X = df_players.select_dtypes(include=[np.number]).drop(columns=['playerId', 'is_riser']).fillna(0)
y = df_players['is_riser']

model.fit(X, y)
riser_importance = pd.Series(model.feature_importances_, index=X.columns)

print("Top KPIs that predict a player will RISE:")
print(riser_importance.sort_values(ascending=False).head(10))

Top KPIs that predict a player will RISE:
SUCCESSFUL_PASSES_BY_FOOT_RIGHT                     0.009595
UNSUCCESSFUL_PASSES_BY_FOOT_RIGHT                   0.008855
SUCCESSFUL_PASSES_BY_ACTION_DIAGONAL_PASS           0.007228
UNSUCCESSFUL_PASSES_BY_FOOT_LEFT                    0.006637
BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS               0.006301
BYPASSED_OPPONENTS_NUMBER_TO_PACKING_ZONE_FBR       0.006051
BALL_WIN_NUMBER_FROM_OPPONENT_PACKING_ZONE_AM       0.006010
SUCCESSFUL_PASSES_TO_PITCH_POSITION_MIDDLE_THIRD    0.005553
OPP_PXT_BLOCK                                       0.005517
NEUTRAL_PASSES_BY_FOOT_RIGHT                        0.005466
dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight

# 1. Split data to have a "Validation Set" for Early Stopping
X_train, X_val, y_train, y_val = train_test_split(X, y_multi, test_size=0.2, stratify=y_multi, random_state=42)

# 2. Calculate weights ONLY for the training portion
train_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 3. Define the model (Remove sample_weight from here)
advanced_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=4,
    objective='multi:softprob',
    random_state=42,
)

# 4. Fit the model properly
advanced_model.fit(
    X_train, y_train,
    sample_weight=train_weights,
    eval_set=[(X_val, y_val)], # The model "tests" itself here during training
    verbose=False
)

Balanced Model trained with Early Stopping.


In [49]:
# Extract importance
importances = advanced_model.feature_importances_
feature_names = X.columns

# Create a clean sorted list
importance_df = pd.DataFrame({'KPI': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("The Advanced Model's Top 10 Decision Drivers:")
print(importance_df.head(10))

The Advanced Model's Top 10 Decision Drivers:
                                                   KPI  Importance
538         REVERSE_PLAY_NUMBER_AT_PHASE_IN_POSSESSION    0.021791
371           UNSUCCESSFUL_PASSES_AT_PHASE_SECOND_BALL    0.020922
464  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_BY_ACTIO...    0.018614
514               SUCCESSFUL_PASSES_TO_PACKING_ZONE_IB    0.013334
172  BALL_WIN_REMOVED_OPPONENTS_FROM_OPPONENT_PACKI...    0.012062
32      BYPASSED_OPPONENTS_FROM_PITCH_POSITION_OWN_BOX    0.011395
266                                  NUMBER_OF_PRESSES    0.010878
163  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_TO_PITCH...    0.010729
492                        NEUTRAL_PASSES_BY_FOOT_LEFT    0.010302
513       BYPASSED_OPPONENTS_NUMBER_TO_PACKING_ZONE_IB    0.010092


In [51]:
# Identify features with zero importance
zero_importance = importance_df[importance_df['Importance'] == 0]

# Identify "Low Impact" features (bottom 10%)
low_importance = importance_df.sort_values(by='Importance')

print(f"Number of features with ZERO importance: {len(zero_importance)}")
print("\nSome of the least important KPIs:")
print(low_importance)

Number of features with ZERO importance: 452

Some of the least important KPIs:
                                                   KPI  Importance
958        SHOT_AT_GOAL_NUMBER_IN_LANE_LEFT_HALF_SPACE    0.000000
923  REVERSE_PLAY_ADDED_OPPONENTS_DEFENDERS_AT_PHAS...    0.000000
922  BALL_WIN_ADDED_TEAMMATES_DEFENDERS_FROM_OPPONE...    0.000000
910    LOST_AERIAL_DUELS_AT_PHASE_ATTACKING_TRANSITION    0.000000
909     WON_GROUND_DUELS_IN_PITCH_POSITION_FINAL_THIRD    0.000000
..                                                 ...         ...
172  BALL_WIN_REMOVED_OPPONENTS_FROM_OPPONENT_PACKI...    0.012062
514               SUCCESSFUL_PASSES_TO_PACKING_ZONE_IB    0.013334
464  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_BY_ACTIO...    0.018614
371           UNSUCCESSFUL_PASSES_AT_PHASE_SECOND_BALL    0.020922
538         REVERSE_PLAY_NUMBER_AT_PHASE_IN_POSSESSION    0.021791

[1007 rows x 2 columns]


In [52]:
# Get probabilities for all 4 classes
probs = advanced_model.predict_proba(X)

# Map the 'PLAYS' probability back to the dataframe
# We find which index corresponds to 'PLAYS'
plays_idx = list(le.classes_).index('PLAYS')
df_players['rise_score'] = probs[:, plays_idx]

# Look for 'Hidden Risers': Status is NOT 'PLAYS' but rise_score is high
hidden_risers = df_players[df_players['status'] != 'PLAYS'].sort_values(by='rise_score', ascending=False)

print("Top 5 'Hidden Risers' (High Rise Score, but haven't moved yet):")
print(hidden_risers[['playerId', 'status', 'rise_score']].head(5))

Top 5 'Hidden Risers' (High Rise Score, but haven't moved yet):
     playerId   status  rise_score
386     63980  DROPPED    0.833840
571    135846   STAYED    0.573158
388     65808  DROPPED    0.406054
96       8182   STAYED    0.402180
511    106974  DROPPED    0.321711


/tmp/ipykernel_5039/3371249176.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_players['rise_score'] = probs[:, plays_idx]


In [55]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Extract raw importance from both models
rf_imp = pd.Series(model.feature_importances_, index=X.columns)
xgb_imp = pd.Series(advanced_model.feature_importances_, index=X.columns)

# 2. Normalize both so they carry equal weight
scaler = MinMaxScaler()
rf_norm = scaler.fit_transform(rf_imp.values.reshape(-1, 1)).flatten()
xgb_norm = scaler.fit_transform(xgb_imp.values.reshape(-1, 1)).flatten()

# 3. Combine into a single DataFrame
consensus_df = pd.DataFrame({
    'KPI': X.columns,
    'RF_Score': rf_norm,
    'XGB_Score': xgb_norm,
    'Total_Score': rf_norm + xgb_norm
})

# 4. Sort by the combined total
consensus_df = consensus_df.sort_values(by='Total_Score', ascending=False)

print("--- The Global Top 10 KPIs (Consensus Model) ---")
print(consensus_df[['KPI', 'Total_Score']].head(10))

--- The Global Top 10 KPIs (Consensus Model) ---
                                                   KPI  Total_Score
471                    SUCCESSFUL_PASSES_BY_FOOT_RIGHT     1.044840
538         REVERSE_PLAY_NUMBER_AT_PHASE_IN_POSSESSION     1.035820
371           UNSUCCESSFUL_PASSES_AT_PHASE_SECOND_BALL     1.029135
464  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_BY_ACTIO...     1.014687
473                       NEUTRAL_PASSES_BY_FOOT_RIGHT     0.972672
474                  UNSUCCESSFUL_PASSES_BY_FOOT_RIGHT     0.953750
475                   UNSUCCESSFUL_PASSES_BY_FOOT_LEFT     0.929703
259                                       DEF_PXT_SHOT     0.898464
492                        NEUTRAL_PASSES_BY_FOOT_LEFT     0.894317
20               BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS     0.818965
